# MNIST Problem

## Imports

In [31]:
import numpy as np
import tensorflow as tf

import tensorflow_datasets as tfds

## Data

In [32]:
# Fetch data from the tensorflow datasets

mnist_dataset,mnist_info = tfds.load(name='MNIST',with_info=True,as_supervised=True)

## Train-Validation-Test

In [33]:
# Splitting the data in train and test
mnist_train,mnist_test = mnist_dataset['train'],mnist_dataset['test']

# Set size of validation data - 10%
validation_size = 0.1*mnist_info.splits['train'].num_examples
validation_size = tf.cast(validation_size,tf.int64)

# Set test data size
test_size = mnist_info.splits['test'].num_examples
test_size = tf.cast(test_size,tf.int64)

## Scaling data

In [34]:
# Defining scale function for map

def scale(image,label):
    image = tf.cast(image,tf.float32)
    image /= 255. # divides each pixel by 255 and returns a float

    return image,label

# Scaling train,validation and test data

scaled_train_validation_data = mnist_train.map(scale)
scaled_test_data = mnist_test.map(scale)


## Preprocessing

In [35]:
Buffer_size = 10000  # To take limited number at a time

# Shuffle data
shuffle_train_val_data = scaled_train_validation_data.shuffle(Buffer_size)

validation_data = shuffle_train_val_data.take(validation_size)
train_data = shuffle_train_val_data.skip(validation_size)

test_data = scaled_test_data

# Creating Batches

Batch_size = 100

train_data = train_data.batch(Batch_size)
validation_data = validation_data.batch(validation_size)
test_data = test_data.batch(test_size)

validation_input,validation_output = next(iter(validation_data))

W0000 00:00:1781550974.562134    4753 cache_dataset_ops.cc:912] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


## Model

### Outline Model

In [36]:
# Layers Size

input_size = 728
output_size = 10
hidden_layer_size = 50

# Model

model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28,28,1)),  # Input Layer
    tf.keras.layers.Dense(hidden_layer_size,activation='relu'), # Hidden Layer 1
    tf.keras.layers.Dense(hidden_layer_size,activation='relu'), # Hidden Layer 2
    tf.keras.layers.Dense(hidden_layer_size,activation='relu'), # Hidden Layer 3
    tf.keras.layers.Dense(output_size,activation='softmax') # Output Layer
])

### Optimizer and Loss Function

In [37]:
model.compile(optimizer='adam',loss = 'sparse_categorical_crossentropy',metrics=['accuracy'])

## Training

In [39]:
NO_OF_EPOCHS = 15
model.fit(train_data,epochs=NO_OF_EPOCHS,validation_data=(validation_input,validation_output),verbose=2)

Epoch 1/15
540/540 - 9s - 17ms/step - accuracy: 0.8841 - loss: 0.4052 - val_accuracy: 0.9418 - val_loss: 0.1959
Epoch 2/15
540/540 - 6s - 11ms/step - accuracy: 0.9500 - loss: 0.1698 - val_accuracy: 0.9585 - val_loss: 0.1439
Epoch 3/15
540/540 - 6s - 10ms/step - accuracy: 0.9627 - loss: 0.1271 - val_accuracy: 0.9680 - val_loss: 0.1082
Epoch 4/15
540/540 - 6s - 12ms/step - accuracy: 0.9697 - loss: 0.1015 - val_accuracy: 0.9738 - val_loss: 0.0892
Epoch 5/15
540/540 - 6s - 11ms/step - accuracy: 0.9744 - loss: 0.0850 - val_accuracy: 0.9752 - val_loss: 0.0805
Epoch 6/15
540/540 - 6s - 10ms/step - accuracy: 0.9775 - loss: 0.0749 - val_accuracy: 0.9762 - val_loss: 0.0749
Epoch 7/15
540/540 - 6s - 11ms/step - accuracy: 0.9803 - loss: 0.0638 - val_accuracy: 0.9783 - val_loss: 0.0746
Epoch 8/15
540/540 - 6s - 12ms/step - accuracy: 0.9827 - loss: 0.0565 - val_accuracy: 0.9832 - val_loss: 0.0618
Epoch 9/15
540/540 - 7s - 12ms/step - accuracy: 0.9844 - loss: 0.0515 - val_accuracy: 0.9825 - val_loss:

## Testing

In [40]:
test_loss,test_accuracy = model.evaluate(test_data)

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.9696 - loss: 0.1194


In [46]:
print(f"Test Loss: {test_loss*100:.2f} %   Test Accuracy: {test_accuracy*100:.2f} % ")

Test Loss: 11.94 %   Test Accuracy: 96.96 % 
